# CDC-CP extension of the ConvNeXt–CLIP ensemble — development only

This is an independent extension of the teammate's `Stage1_CDCCP.ipynb`, using his exact `cdc-cp_code_final_model.zip` and rerun **Stage 1** checkpoints. It never renames or overwrites his result directories, never reads the official test CSVs, and never uses Stage 2 weights (which saw the validation images). The reference is his P3 ensemble: crop = ConvNeXt-Tiny + CLIP, disease = CLIP + ConvNeXt-Small, equal mean log-probabilities, raw pair scoring, gamma=0, all 9,625 pairs. This P3 naming is different from the original EfficientNet CDC-CP P3 prototype method.

Run E0–E5 in order. E2 extracts cached features and may be resumed per arm; E4 trains inexpensive heads on the same cached two-view features. E6–E7 are a more expensive **optional matched online continuation** (frozen vs unfrozen, same parent heads and data). No official-test or full-8000 continuation is included. This is a proposed experiment, not a demonstrated accuracy improvement.

Before starting: in Google Drive add a shortcut to the teammate's **DPD-Small** shared folder in My Drive (if there is already a folder with that name, give the shortcut a distinct name, e.g. `DPD-Small-Norman` and adjust `SOURCE` below). Enable an A100/L4 GPU. All outputs are written to a separate folder in your own My Drive. Do not Run All on the original teammate notebook.

In [3]:
from pathlib import Path

mount = Path("/content/drive")
folder_id = "1jbwvz6-BATK6XBUHh0ho1eYtJq0qFk7a"
target = mount / ".shortcut-targets-by-id" / folder_id
my_drive = mount / "MyDrive"

split_names = (
    "DPD_Small_train.csv",
    "DPD_Small_test_seen.csv",
    "DPD_Small_test_unseen.csv",
)

candidates = [
    target / "DPD-Small",
    target,
    my_drive / "DPD-Small",
    *sorted(my_drive.glob("DPD-Small*")),
]

DATA_ROOT = next(
    (
        path for path in candidates
        if (path / "DPD_Small_dataset").is_dir()
        and all((path / name).is_file() for name in split_names)
    ),
    None,
)

if DATA_ROOT is None:
    for path in dict.fromkeys(candidates):
        children = sorted(p.name for p in path.iterdir()) if path.is_dir() else []
        print(f"{path}: {children[:20] if path.is_dir() else 'not visible'}")
    raise FileNotFoundError("Colab could not locate the dataset and all three splits.")

IMAGE_ROOT = DATA_ROOT / "DPD_Small_dataset"
TRAIN_CSV, SEEN_CSV, UNSEEN_CSV = (DATA_ROOT / name for name in split_names)

print("Dataset ready:", DATA_ROOT)

Dataset ready: /content/drive/.shortcut-targets-by-id/1jbwvz6-BATK6XBUHh0ho1eYtJq0qFk7a/DPD-Small


In [7]:
# @title E0 · Read-only source paths and locked development protocol

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys, os, json, hashlib, shutil, zipfile, math, random, gc, inspect
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import transforms as T
from PIL import Image
from torch.utils.data import Dataset, DataLoader, TensorDataset
from IPython.display import display

# Correct Google Drive paths
MYDRIVE = Path('/content/drive/MyDrive')

SOURCE = Path('/content/drive/.shortcut-targets-by-id/1jbwvz6-BATK6XBUHh0ho1eYtJq0qFk7a/DPD-Small')

RERUN = SOURCE / 'stage1_weights_rerun'
SAVE = MYDRIVE / 'CDC_CP_ConvNeXt_CLIP_Extension_v1'

LOCAL = Path('/content/cdc_cp_extension_v1')
CODE = LOCAL / 'code' / 'cdc-cp'
IMAGES = LOCAL / 'images'

ARMS = ['crop_b1', 'crop_clip', 'disease_clip', 'disease_small_reg']

TASKS = {
    'crop': ['crop_b1', 'crop_clip'],
    'disease': ['disease_clip', 'disease_small_reg']
}

# Experimental settings
SEED = 42
CACHE_BATCH = 32
ONLINE_BATCH = 8
WORKERS = 2

HEAD_EPOCHS = 12
HEAD_REPORT_EPOCHS = (8, 12)
ONLINE_EPOCHS = 3
HEAD_BATCH = 128

HEAD_LR = 1e-3
BACKBONE_LR = 1e-6
WEIGHT_DECAY = 1e-4

SUPCON_WEIGHT = 0.1
TAU = 0.07
PROTOTYPE_TAU = 0.07

RUN_ONLINE = False
DEVICE = torch.device('cuda')

assert torch.cuda.is_available(), 'Select GPU in Runtime > Change runtime type'

assert SAVE.resolve() != SOURCE.resolve()
assert not SAVE.resolve().is_relative_to(SOURCE.resolve())

# Verify all required files
required = [
    SOURCE / 'cdc-cp_code_final_model.zip',
    SOURCE / 'DPD_Small_train.csv',
    SOURCE / 'DPD_Small_dataset'
] + [
    RERUN / 'stage1' / a / 'best.pt'
    for a in ARMS
]

for p in required:
    print('[OK]' if p.exists() else '[MISSING]', p.name)

missing = [str(p) for p in required if not p.exists()]

assert not missing, f'Missing teammate files: {missing}'

SAVE.mkdir(parents=True, exist_ok=True)
LOCAL.mkdir(parents=True, exist_ok=True)

print('\nSource (read only):', SOURCE)
print('Output:', SAVE)
print('GPU:', torch.cuda.get_device_name(0))
print('\nE0 completed successfully.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[OK] cdc-cp_code_final_model.zip
[OK] DPD_Small_train.csv
[OK] DPD_Small_dataset
[OK] best.pt
[OK] best.pt
[OK] best.pt
[OK] best.pt

Source (read only): /content/drive/.shortcut-targets-by-id/1jbwvz6-BATK6XBUHh0ho1eYtJq0qFk7a/DPD-Small
Output: /content/drive/MyDrive/CDC_CP_ConvNeXt_CLIP_Extension_v1
GPU: NVIDIA L4

E0 completed successfully.


In [8]:
# @title E1 · Load original project code, verify split and checkpoint provenance
# Extraction is into a fresh local folder; no rename/delete of shared outputs.
if not CODE.exists():
    (LOCAL/'code').mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(SOURCE/'cdc-cp_code_final_model.zip') as z:
        z.extractall(LOCAL/'code')
assert (CODE/'cdccp'/'final_model.py').is_file()
if str(CODE) not in sys.path: sys.path.insert(0,str(CODE))
try:
    from transformers import CLIPVisionConfig, CLIPVisionModelWithProjection
except ImportError as exc:
    raise ImportError('In a new cell run %pip install transformers, then restart the runtime and rerun E0–E1.') from exc
from cdccp import final_model as fm, single_task as st
from cdccp.losses import supcon
from cdccp.colab_io import copy_tree
assert fm.code_fingerprint(CODE)=='0777e2f70c39689ae087dcf501d8aca86cf7d1c87d6ab97483aec6a18787c864', 'Wrong original code version'
SPLITS=CODE/'data'/'splits_v2'
fm.check_inputs(SOURCE/'DPD_Small_train.csv',SPLITS)
train_fit,val_seen,val_unseen=st.load_dev(SPLITS)
FRAMES={'fit':train_fit,'seen':val_seen,'unseen':val_unseen}
CROP_IDS,DISEASE_IDS=st.class_ids(*FRAMES.values())
assert CROP_IDS==list(range(55)) and DISEASE_IDS==list(range(175))
assert (len(train_fit),len(val_seen),len(val_unseen))==(6905,695,400)
assert set(val_unseen.composition_id).isdisjoint(set(train_fit.composition_id))
# Copy only images, not official labels. Files are flat in original dataset folder.
copy_tree(SOURCE/'DPD_Small_dataset',IMAGES)
for name,df in FRAMES.items():
    missing=[f for f in df.filename if not (IMAGES/str(f)).is_file()]
    assert not missing, (name,missing[:5])
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for block in iter(lambda:f.read(2**20),b''): h.update(block)
    return h.hexdigest()
def atomic_torch(obj,path):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp');torch.save(obj,tmp);os.replace(tmp,path)
def atomic_json(obj,path):
    path=Path(path);tmp=path.with_suffix(path.suffix+'.tmp');tmp.write_text(json.dumps(obj,indent=2));os.replace(tmp,path)
CKPTS={a:RERUN/'stage1'/a/'best.pt' for a in ARMS}
EXPECTED_EPOCHS={'crop_b1':10,'crop_clip':9,'disease_clip':13,'disease_small_reg':21}
PROVENANCE={}
for a,p in CKPTS.items():
    ck=torch.load(p,map_location='cpu',weights_only=True)
    assert ck['arm']==a and ck['epoch']==EXPECTED_EPOCHS[a]
    assert ck['crop_ids']==CROP_IDS and ck['disease_ids']==DISEASE_IDS
    assert ck['backbone']==st.ARMS[a].backbone
    PROVENANCE[a]={'sha256':sha(p),'epoch':int(ck['epoch']),'task':ck['task'],'backbone':ck['backbone']}
    del ck
PROTO={'code_sha':fm.code_fingerprint(CODE),'splits':{x:sha(SPLITS/x) for x in ['train_fit.csv','val_seen.csv','val_unseen.csv']},'checkpoints':PROVENANCE,'seed':SEED,'head_epochs':HEAD_EPOCHS,'report_epochs':HEAD_REPORT_EPOCHS,'head_batch':HEAD_BATCH,'head_lr':HEAD_LR,'wd':WEIGHT_DECAY,'supcon_weight':SUPCON_WEIGHT,'tau':TAU,'prototype_tau':PROTOTYPE_TAU,'online_epochs':ONLINE_EPOCHS,'online_batch':ONLINE_BATCH,'backbone_lr':BACKBONE_LR,'train_res':224,'conv_eval_res':[224,320],'clip_eval_res':224,'flip':True,'gamma':0,'candidates':9625,'official_test_access':False,'cache_views':'2 fixed augmented views; normal+flip evaluation'}
# JSON roundtrip normalises tuples for comparison
PROTO=json.loads(json.dumps(PROTO)); MANIFEST=SAVE/'protocol.json'
if MANIFEST.exists(): assert json.loads(MANIFEST.read_text())==PROTO, 'Protocol changed; choose a NEW output folder rather than overwrite an experiment.'
else: atomic_json(PROTO,MANIFEST)
print('Verified 6905/695/400, source code and four Stage 1 weights. No official labels read.')
display(pd.DataFrame(PROVENANCE).T)

copied 11448 of 11448 files from /content/drive/.shortcut-targets-by-id/1jbwvz6-BATK6XBUHh0ho1eYtJq0qFk7a/DPD-Small/DPD_Small_dataset to /content/cdc_cp_extension_v1/images in 376 s (sizes checked)
Verified 6905/695/400, source code and four Stage 1 weights. No official labels read.


,sha256,epoch,task,backbone
crop_b1,64509062a54291eeee16ad89d1f0f08c5001875ae2e4bf...,10,crop,convnext_tiny
crop_clip,8de8e25ca8bdb4d3db0257c9fe83378daf7b319eb0bed7...,9,crop,clip_vit_b16
disease_clip,6159538736a770ba3fa75af47ff18d9d0e1cc7616452fc...,13,disease,clip_vit_b16
disease_small_reg,54907b10c7f3be4a5ccff9205baf0fbc959bf1ddbbf869...,21,disease,convnext_small


In [9]:
# @title E2 · Cache frozen features; reproduce original ensemble at 224 and test ConvNeXt 320
# Exact original arm-specific normalisation and training augmentations. CLIP remains at 224.
CACHE=SAVE/'features';CACHE.mkdir(exist_ok=True)
AMP_DTYPE=torch.float16

def base_model(a):
    return st.load_trained(CKPTS[a],DEVICE)

def is_clip(a): return st.ARMS[a].backbone=='clip_vit_b16'
def features(base,x,a):
    if is_clip(a): return base.vision(pixel_values=x).image_embeds.float()
    h=base.avgpool(base.features(x))
    return base.classifier[1](base.classifier[0](h)).float()
def original_logits(base,f,a):
    if is_clip(a): return base.scale()*F.normalize(f.float(),dim=-1) @ base.text_features.T
    return base.classifier[-1](f)
def transform(a,train=False,res=224):
    norm=st.NORMS.get(st.ARMS[a].backbone)
    if train: return st.train_transform(st.ARMS[a].aug,norm)
    if res==224: return st.eval_transform(norm)
    assert not is_clip(a) and res==320
    return T.Compose([T.Resize(366),T.CenterCrop(320),T.ToTensor(),T.Normalize(*(norm or (st.MEAN,st.STD)))])
class Views(Dataset):
    def __init__(self,df,a,train=False,res=224): self.df=df.reset_index(drop=True); self.tf=transform(a,train,res);self.train=train
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        with Image.open(IMAGES/str(r.filename)) as im:
            im=im.convert('RGB');x=self.tf(im);v=self.tf(im) if self.train else x.flip(-1)
        return torch.stack((x,v)),int(r.crop_id),int(r.disease_id)
def loader(df,a,train=False,res=224,batch=CACHE_BATCH,shuffle=False,seed=SEED):
    return DataLoader(Views(df,a,train,res),batch_size=batch,shuffle=shuffle,num_workers=WORKERS,pin_memory=True,generator=torch.Generator().manual_seed(seed))
def flip_logp(logits):
    return torch.logsumexp(F.log_softmax(logits.float(),dim=-1),dim=1)-math.log(2)
@torch.no_grad()
def cache_part(base,a,df,train,res):
    base.eval();fs=[];lp=[];cs=[];ds=[]
    for x,c,d in loader(df,a,train,res):
        b=x.shape[0];x=x.flatten(0,1).to(DEVICE,non_blocking=True)
        with torch.autocast('cuda',dtype=AMP_DTYPE): f=features(base,x,a)
        # Original scoring head in FP32, using the same normal+flip inputs.
        logits=original_logits(base,f,a).reshape(b,2,-1)
        assert torch.isfinite(f).all() and torch.isfinite(logits).all()
        fs.append(f.reshape(b,2,-1).cpu());lp.append(flip_logp(logits).cpu());cs.append(c);ds.append(d)
    return {'f':torch.cat(fs),'logp':torch.cat(lp),'crop':torch.cat(cs),'disease':torch.cat(ds),'filenames':df.filename.tolist()}
def load_cache(a):
    data=torch.load(CACHE/f'{a}.pt',map_location='cpu',weights_only=True)
    assert data['sha']==PROVENANCE[a]['sha256']
    for k,v in data['parts'].items():
        split=k.split('_')[0];assert v['filenames']==FRAMES[split].filename.tolist()
    return data['parts']
for a in ARMS:
    dest=CACHE/f'{a}.pt'
    if dest.exists(): load_cache(a);print('Reuse verified cache:',a);continue
    st.seed_all(SEED);base=base_model(a)
    # Verify the extracted feature path exactly reproduces the original forward in FP32.
    sample,_,_=next(iter(loader(val_seen.iloc[:2],a,False,224,batch=2)))
    sample=sample[:,0].to(DEVICE)
    with torch.no_grad():
        direct=base(sample).float();reconstructed=original_logits(base,features(base,sample,a),a).float()
        assert torch.allclose(direct,reconstructed,rtol=1e-4,atol=1e-4), f'{a}: feature path mismatch'
    print(a,'original forward equivalence: PASS')
    del sample,direct,reconstructed
    parts={'fit_aug224':cache_part(base,a,train_fit,True,224),'fit_eval224':cache_part(base,a,train_fit,False,224)}
    for split in ('seen','unseen'):
        for res in ([224] if is_clip(a) else [224,320]):
            parts[f'{split}_eval{res}']=cache_part(base,a,FRAMES[split],False,res)
    if not is_clip(a): parts['fit_eval320']=cache_part(base,a,train_fit,False,320)
    atomic_torch({'sha':PROVENANCE[a]['sha256'],'parts':parts},dest)
    del parts,base;gc.collect();torch.cuda.empty_cache();print('Cached',a)
print('Feature caches saved. Training uses only fit_aug224; prototype banks use only fit_eval data.')

crop_b1 original forward equivalence: PASS
Cached crop_b1
crop_clip original forward equivalence: PASS
Cached crop_clip
disease_clip original forward equivalence: PASS
Cached disease_clip
disease_small_reg original forward equivalence: PASS
Cached disease_small_reg
Feature caches saved. Training uses only fit_aug224; prototype banks use only fit_eval data.


In [10]:
# @title E3 · Original ensemble control and resolution ablation (no training)
from collections import defaultdict

def part_key(a,split,res): return f'{split}_eval{224 if is_clip(a) else res}'
def fuse(logps,split):
    return tuple(np.mean([np.asarray(logps[a,split]) for a in TASKS[t]],axis=0) for t in ('crop','disease'))
def report(name,logps,epoch=None):
    result={}
    for split in ('seen','unseen'):
        c,d=fuse(logps,split);df=FRAMES[split]
        m=st.composition_metrics(c,d,df.crop_id.to_numpy(),df.disease_id.to_numpy())
        result.update({f'{split}_{k}':v*100 for k,v in m.items()})
    for k in ('composition_top1','composition_top5','crop_top1','disease_top1'):
        result[f'{k}_HM']=st.harmonic(result[f'seen_{k}'],result[f'unseen_{k}'])
    return {'variant':name,'epoch':epoch,**result}

def original_logs(res):
    out={}
    for a in ARMS:
        cache=load_cache(a)
        for s in ('seen','unseen'): out[a,s]=cache[part_key(a,s,res)]['logp'].numpy()
        del cache
    return out
BASE224=original_logs(224);BASE320=original_logs(320)
rows=[report('Original ensemble 224',BASE224),report('Original ensemble ConvNeXt320 / CLIP224',BASE320)]
# Compare exact saved predictions when accessible (same split order, same four arms).
archived=RERUN/'results'/'dev_flip_logprobs.npz'
if archived.is_file():
    with np.load(archived,allow_pickle=False) as saved:
        exact={(a,s):saved[f'{a}_{s}'] for a in ARMS for s in ('seen','unseen')}
    assert all(exact[a,s].shape==BASE224[a,s].shape for a,s in exact)
    rows.insert(0,report('Archived Stage1 ensemble (original predictions)',exact))
    for a,s in exact:
        print(a,s,'argmax agreement with rerun:',round(100*np.mean(exact[a,s].argmax(1)==BASE224[a,s].argmax(1)),2),'%')
else: print('Archived log probabilities unavailable; do not claim bit-exact reproduction.')
BASE_ROWS=rows
pd.DataFrame(rows).to_csv(SAVE/'reference_resolution.csv',index=False)
SHOW=['variant','epoch','seen_composition_top1','unseen_composition_top1','composition_top1_HM','composition_top5_HM','unseen_disease_top1']
display(pd.DataFrame(rows)[SHOW].round(2))

crop_b1 seen argmax agreement with rerun: 99.86 %
crop_b1 unseen argmax agreement with rerun: 100.0 %
crop_clip seen argmax agreement with rerun: 100.0 %
crop_clip unseen argmax agreement with rerun: 100.0 %
disease_clip seen argmax agreement with rerun: 99.86 %
disease_clip unseen argmax agreement with rerun: 100.0 %
disease_small_reg seen argmax agreement with rerun: 100.0 %
disease_small_reg unseen argmax agreement with rerun: 100.0 %


,variant,epoch,seen_composition_top1,unseen_composition_top1,composition_top1_HM,composition_top5_HM,unseen_disease_top1
0,Archived Stage1 ensemble (original predictions),None,79.42,10.25,18.16,32.41,31.0
1,Original ensemble 224,None,79.42,10.25,18.16,32.41,31.0
2,Original ensemble ConvNeXt320 / CLIP224,None,79.28,9.25,16.57,33.41,30.0


In [11]:
# @title E4 · Matched frozen-head experiments: CE shallow, CE deep, CE+SupCon deep
# All heads share identical cached views, labels, optimizer and fixed epoch budget.
# These are new concept heads, not a claim of isolating depth vs the original classifier.
HEADS=SAVE/'heads';HEADS.mkdir(exist_ok=True)
VARIANTS={'shallow_ce':('shallow',0.),'deep_ce':('deep',0.),'deep_supcon':('deep',SUPCON_WEIGHT)}
class ConceptHead(nn.Module):
    def __init__(self,dim,n,depth):
        super().__init__();dims=[dim,256,128] if depth=='shallow' else [dim,512,256,128]
        layers=[]
        for i,(di,do) in enumerate(zip(dims,dims[1:])):
            layers.append(nn.Linear(di,do))
            if i<len(dims)-2:layers.extend([nn.ReLU(),nn.Dropout(.2)])
        self.project=nn.Sequential(*layers);self.classifier=nn.Linear(128,n)
    def forward(self,f):
        raw=self.project(f.float());return self.classifier(raw),F.normalize(raw.float(),dim=-1)
def task_of(a):return st.ARMS[a].task
def nclasses(a):return 55 if task_of(a)=='crop' else 175
def make_head(a,depth,dim):return ConceptHead(dim,nclasses(a),depth).to(DEVICE)
def objective(head,f,y,a,w):
    b=f.shape[0]; logits,z=head(f.flatten(0,1)); yy=y.repeat_interleave(2)
    ce=F.cross_entropy(logits,yy,label_smoothing=st.RECIPES[st.ARMS[a].recipe]['label_smoothing'])
    con=supcon(z,yy,tau=TAU) if w else ce.detach()*0
    return ce+w*con,ce,con
for a in ARMS:
    cache=load_cache(a);part=cache['fit_aug224']; dim=part['f'].shape[-1]
    x=part['f'];y=part[task_of(a)]
    for name,(depth,w) in VARIANTS.items():
        st.seed_all(SEED); head=make_head(a,depth,dim);opt=torch.optim.AdamW(head.parameters(),lr=HEAD_LR,weight_decay=WEIGHT_DECAY)
        folder=HEADS/name/a;folder.mkdir(parents=True,exist_ok=True)
        # Resume from last completed epoch; fixed epochs 8 and 12 retained for comparison.
        last=folder/'last.pt';start=1;hist=[]
        if last.exists():
            state=torch.load(last,map_location='cpu',weights_only=True);assert state['sha']==PROVENANCE[a]['sha256']
            head.load_state_dict(state['head']);opt.load_state_dict(state['optimizer']);start=state['epoch']+1;hist=state['history']
        for epoch in range(start,HEAD_EPOCHS+1):
            st.seed_all(SEED+epoch);head.train(); sums=np.zeros(3);n=0
            dl=DataLoader(TensorDataset(x,y),batch_size=HEAD_BATCH,shuffle=True,generator=torch.Generator().manual_seed(SEED+epoch))
            for f,label in dl:
                f,label=f.to(DEVICE),label.to(DEVICE);opt.zero_grad(set_to_none=True)
                loss,ce,con=objective(head,f,label,a,w);loss.backward();opt.step()
                sums+=np.array([loss.item(),ce.item(),con.item()])*len(label);n+=len(label)
            hist.append({'epoch':epoch,'loss':float(sums[0]/n),'ce':float(sums[1]/n),'supcon':float(sums[2]/n)})
            state={'sha':PROVENANCE[a]['sha256'],'epoch':epoch,'dim':dim,'depth':depth,'weight':w,'head':head.state_dict(),'optimizer':opt.state_dict(),'history':hist}
            if epoch in HEAD_REPORT_EPOCHS: atomic_torch(state,folder/f'epoch{epoch:02d}.pt')
            atomic_torch(state,last);pd.DataFrame(hist).to_csv(folder/'history.csv',index=False)
        print(a,name,'completed',HEAD_EPOCHS,'epochs');del head,opt
    del cache,part,x,y;gc.collect();torch.cuda.empty_cache()

crop_b1 shallow_ce completed 12 epochs
crop_b1 deep_ce completed 12 epochs
crop_b1 deep_supcon completed 12 epochs
crop_clip shallow_ce completed 12 epochs
crop_clip deep_ce completed 12 epochs
crop_clip deep_supcon completed 12 epochs
disease_clip shallow_ce completed 12 epochs
disease_clip deep_ce completed 12 epochs
disease_clip deep_supcon completed 12 epochs
disease_small_reg shallow_ce completed 12 epochs
disease_small_reg deep_ce completed 12 epochs
disease_small_reg deep_supcon completed 12 epochs


In [12]:
# @title E5 · Evaluate classifier and train-only prototype CDC-CP, fixed epochs and two resolutions
# Prototypes are class-mean normalised concept vectors. Pair scores remain additive/factorised.
# No learned pair interaction is implied. Compare prototype to classifier on SAME head/checkpoint.
@torch.no_grad()
def head_outputs(head,feat):
    head.eval();ls=[];zs=[]
    for f in feat.split(256):
        b=len(f);l,z=head(f.flatten(0,1).to(DEVICE));ls.append(l.reshape(b,2,-1).cpu());zs.append(z.reshape(b,2,-1).cpu())
    return torch.cat(ls),torch.cat(zs)
def build_bank(z,y,n):
    # Use only unflipped, deterministic training view; no validation features in bank.
    z=z[:,0,:].float(); sums=torch.zeros(n,z.shape[-1]);counts=torch.bincount(y,minlength=n)
    assert (counts>0).all(), 'Every class must have a training example'
    sums.index_add_(0,y,z);return F.normalize(sums/counts[:,None],dim=-1)
def head_logprobs(head,cache,a,res):
    train=cache[part_key(a,'fit',res)];_,z=head_outputs(head,train['f']);bank=build_bank(z,train[task_of(a)],nclasses(a))
    result={}
    for s in ('seen','unseen'):
        part=cache[part_key(a,s,res)];l,z=head_outputs(head,part['f'])
        result[s]={'classifier':flip_logp(l).numpy(),'prototype':flip_logp((z@bank.T)/PROTOTYPE_TAU).numpy()}
    return result,bank
results=list(BASE_ROWS);heads_logs={}
for name,(depth,w) in VARIANTS.items():
    for epoch in HEAD_REPORT_EPOCHS:
        for res in (224,320):
            logs={'classifier':{},'prototype':{}}
            for a in ARMS:
                cache=load_cache(a);state=torch.load(HEADS/name/a/f'epoch{epoch:02d}.pt',map_location='cpu',weights_only=True)
                assert state['sha']==PROVENANCE[a]['sha256'];head=make_head(a,depth,state['dim']);head.load_state_dict(state['head'])
                pred,bank=head_logprobs(head,cache,a,res)
                for s in ('seen','unseen'):
                    for mode in logs:logs[mode][a,s]=pred[s][mode]
                atomic_torch({'bank':bank,'sha':state['sha'],'epoch':epoch,'resolution':224 if is_clip(a) else res},HEADS/name/a/f'prototype_e{epoch:02d}_r{res}.pt')
                del cache,head,state
            suffix=f'ConvNeXt{res}/CLIP224'
            results.append(report(f'{name} classifier {suffix}',logs['classifier'],epoch))
            results.append(report(f'{name} prototypes {suffix}',logs['prototype'],epoch))
            if name=='deep_supcon' and epoch==HEAD_REPORT_EPOCHS[-1]:
                base=BASE224 if res==224 else BASE320
                hybrid={key:.5*base[key]+.5*logs['prototype'][key] for key in base}
                results.append(report(f'Original + CDC-CP prototype 50/50 {suffix}',hybrid,epoch))
            heads_logs[(name,epoch,res)]=logs
            np.savez_compressed(SAVE/f'{name}_e{epoch:02d}_r{res}_predictions.npz',**{f'{m}_{a}_{s}':v for m,items in logs.items() for (a,s),v in items.items()})
summary=pd.DataFrame(results);summary.to_csv(SAVE/'development_comparison.csv',index=False)
display(summary[SHOW].round(2))
print('All results are development validation, not official test accuracy.')

,variant,epoch,seen_composition_top1,unseen_composition_top1,composition_top1_HM,composition_top5_HM,unseen_disease_top1
0,Archived Stage1 ensemble (original predictions),NaN,79.42,10.25,18.16,32.41,31.00
1,Original ensemble 224,NaN,79.42,10.25,18.16,32.41,31.00
2,Original ensemble ConvNeXt320 / CLIP224,NaN,79.28,9.25,16.57,33.41,30.00
3,shallow_ce classifier ConvNeXt224/CLIP224,8.0,77.41,9.50,16.92,29.92,29.00
4,shallow_ce prototypes ConvNeXt224/CLIP224,8.0,73.38,8.00,14.43,27.39,28.00
5,shallow_ce classifier ConvNeXt320/CLIP224,8.0,76.12,8.75,15.70,30.58,28.75
6,shallow_ce prototypes ConvNeXt320/CLIP224,8.0,73.67,7.25,13.20,26.68,27.50
7,shallow_ce classifier ConvNeXt224/CLIP224,12.0,78.71,9.50,16.95,28.53,29.25
8,shallow_ce prototypes ConvNeXt224/CLIP224,12.0,75.54,8.25,14.88,28.50,27.50
9,shallow_ce classifier ConvNeXt320/CLIP224,12.0,77.55,9.00,16.13,29.21,30.00


All results are development validation, not official test accuracy.


In [15]:
RUN_ONLINE = True

## Optional E6–E7 — online fine-tuning experiment

The cached-feature experiment is the inexpensive first step. To test the prior full-fine-tuning finding fairly, both online arms start from the **same epoch-12 frozen Deeper SupCon head and exact Stage 1 backbone** and train for three further fixed epochs with fresh two-view augmentation, same seed, data order and optimizer/head learning rate. Frozen vs full backbone trainability is the deliberate intervention (normal frozen/eval vs train mode). Both branches get the same budget. The full arm updates the image feature extractors and concept heads, not the original unused classifier/text prompt head. Each member is trained independently; this is a **single-task ensemble CDC-CP extension**, not the original shared-backbone architecture. Saves one resumable checkpoint per member per arm (full model + optimizer is large). You need sufficient Drive space. Set `RUN_ONLINE=True` in a separate cell before running E6. Do not change the locked training settings after seeing results; new trials require a new output folder.

In [16]:
# @title E6 · Matched online frozen vs full end-to-end continuation (opt-in)
ONLINE=SAVE/'online';ONLINE.mkdir(exist_ok=True)
if not RUN_ONLINE:
    print('Skipped. To run, set RUN_ONLINE=True in a new cell, then rerun E6 and E7.')
else:
    for a in ARMS:
        parent=torch.load(HEADS/'deep_supcon'/a/f'epoch{HEAD_EPOCHS:02d}.pt',map_location='cpu',weights_only=True)
        assert parent['sha']==PROVENANCE[a]['sha256']
        for mode in ('frozen_online','full_online'):
            st.seed_all(SEED); base=base_model(a);head=make_head(a,'deep',parent['dim']);head.load_state_dict(parent['head'])
            full=mode=='full_online';base.requires_grad_(full)
            # Original prediction heads are unused by the extension.
            if is_clip(a):base.logit_scale.requires_grad_(False)
            else:base.classifier[-1].requires_grad_(False)
            groups=[{'params':head.parameters(),'lr':HEAD_LR}]
            if full:groups.insert(0,{'params':[p for p in base.parameters() if p.requires_grad],'lr':BACKBONE_LR})
            opt=torch.optim.AdamW(groups,weight_decay=WEIGHT_DECAY);scaler=torch.amp.GradScaler('cuda')
            folder=ONLINE/mode/a;folder.mkdir(parents=True,exist_ok=True);last=folder/'last.pt';start=1;hist=[]
            if last.exists():
                state=torch.load(last,map_location='cpu',weights_only=True);assert state['sha']==PROVENANCE[a]['sha256']
                if full:base.load_state_dict(state['base'])
                head.load_state_dict(state['head']);opt.load_state_dict(state['optimizer']);scaler.load_state_dict(state['scaler']);start=state['epoch']+1;hist=state['history'];del state
            for epoch in range(start,ONLINE_EPOCHS+1):
                st.seed_all(SEED+epoch);base.train(full);head.train();tot=0.;n=0
                for x,c,d in loader(train_fit,a,True,224,batch=ONLINE_BATCH,shuffle=True,seed=SEED+epoch):
                    y=(c if task_of(a)=='crop' else d).to(DEVICE);b=len(y);x=x.flatten(0,1).to(DEVICE,non_blocking=True)
                    opt.zero_grad(set_to_none=True)
                    with torch.set_grad_enabled(full):
                        with torch.autocast('cuda',dtype=AMP_DTYPE): f=features(base,x,a)
                    f=f.reshape(b,2,-1)
                    loss,ce,con=objective(head,f,y,a,SUPCON_WEIGHT)
                    scaler.scale(loss).backward();scaler.unscale_(opt);torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g['params']],1.0);scaler.step(opt);scaler.update()
                    tot+=loss.item()*b;n+=b
                hist.append({'epoch':epoch,'loss':tot/n})
                state={'sha':PROVENANCE[a]['sha256'],'epoch':epoch,'dim':parent['dim'],'head':head.state_dict(),'optimizer':opt.state_dict(),'scaler':scaler.state_dict(),'history':hist}
                if full:state['base']=base.state_dict()
                atomic_torch(state,last);pd.DataFrame(hist).to_csv(folder/'history.csv',index=False);del state
                print(a,mode,epoch,round(tot/n,4))
            del opt,scaler,head,base;gc.collect();torch.cuda.empty_cache()
        del parent

crop_b1 frozen_online 1 0.1781
crop_b1 frozen_online 2 0.1579
crop_b1 frozen_online 3 0.147
crop_b1 full_online 1 0.2012
crop_b1 full_online 2 0.1831
crop_b1 full_online 3 0.1593
crop_clip frozen_online 1 0.1321
crop_clip frozen_online 2 0.1188
crop_clip frozen_online 3 0.0915
crop_clip full_online 1 0.1257
crop_clip full_online 2 0.1031
crop_clip full_online 3 0.0868
disease_clip frozen_online 1 0.2725
disease_clip frozen_online 2 0.1907
disease_clip frozen_online 3 0.1699
disease_clip full_online 1 0.2829
disease_clip full_online 2 0.1805
disease_clip full_online 3 0.1444
disease_small_reg frozen_online 1 1.4866
disease_small_reg frozen_online 2 1.4263
disease_small_reg frozen_online 3 1.383
disease_small_reg full_online 1 1.5979
disease_small_reg full_online 2 1.5156
disease_small_reg full_online 3 1.4697


In [17]:
# @title E7 · Evaluate both online continuation arms (development only)
if not RUN_ONLINE:
    print('Skipped online evaluation.')
else:
    outrows=[]
    for mode in ('frozen_online','full_online'):
        for res in (224,320):
            logs={'classifier':{},'prototype':{}}
            for a in ARMS:
                state=torch.load(ONLINE/mode/a/'last.pt',map_location='cpu',weights_only=True)
                assert state['epoch']==ONLINE_EPOCHS and state['sha']==PROVENANCE[a]['sha256']
                head=make_head(a,'deep',state['dim']);head.load_state_dict(state['head'])
                if mode=='frozen_online': cache=load_cache(a)
                else:
                    base=base_model(a);base.load_state_dict(state['base']);base.eval();r=224 if is_clip(a) else res
                    cache={part_key(a,s,res):cache_part(base,a,FRAMES[s],False,r) for s in ('fit','seen','unseen')}
                    del base
                pred,bank=head_logprobs(head,cache,a,res)
                for s in ('seen','unseen'):
                    for m in logs: logs[m][a,s]=pred[s][m]
                atomic_torch({'bank':bank,'sha':state['sha'],'epoch':state['epoch'],'resolution':224 if is_clip(a) else res},ONLINE/mode/a/f'prototype_r{res}.pt')
                del cache,head,state;gc.collect();torch.cuda.empty_cache()
            for m,lp in logs.items():outrows.append(report(f'{mode} {m} ConvNeXt{res}/CLIP224',lp,ONLINE_EPOCHS))
            base=BASE224 if res==224 else BASE320
            outrows.append(report(f'Original + {mode} prototype 50/50 ConvNeXt{res}/CLIP224',{k:.5*base[k]+.5*logs['prototype'][k] for k in base},ONLINE_EPOCHS))
            np.savez_compressed(SAVE/f'{mode}_r{res}_predictions.npz',**{f'{m}_{a}_{s}':v for m,items in logs.items() for (a,s),v in items.items()})
    combined=pd.concat([pd.read_csv(SAVE/'development_comparison.csv'),pd.DataFrame(outrows)],ignore_index=True)
    combined.to_csv(SAVE/'development_comparison_with_online.csv',index=False)
    display(combined[SHOW].round(2))

,variant,epoch,seen_composition_top1,unseen_composition_top1,composition_top1_HM,composition_top5_HM,unseen_disease_top1
0,Archived Stage1 ensemble (original predictions),NaN,79.42,10.25,18.16,32.41,31.00
1,Original ensemble 224,NaN,79.42,10.25,18.16,32.41,31.00
2,Original ensemble ConvNeXt320 / CLIP224,NaN,79.28,9.25,16.57,33.41,30.00
3,shallow_ce classifier ConvNeXt224/CLIP224,8.0,77.41,9.50,16.92,29.92,29.00
4,shallow_ce prototypes ConvNeXt224/CLIP224,8.0,73.38,8.00,14.43,27.39,28.00
5,shallow_ce classifier ConvNeXt320/CLIP224,8.0,76.12,8.75,15.70,30.58,28.75
6,shallow_ce prototypes ConvNeXt320/CLIP224,8.0,73.67,7.25,13.20,26.68,27.50
7,shallow_ce classifier ConvNeXt224/CLIP224,12.0,78.71,9.50,16.95,28.53,29.25
8,shallow_ce prototypes ConvNeXt224/CLIP224,12.0,75.54,8.25,14.88,28.50,27.50
9,shallow_ce classifier ConvNeXt320/CLIP224,12.0,77.55,9.00,16.13,29.21,30.00


In [19]:

# @title E8 · Lock CDC-CP configuration before official testing

import re
import json

official_dir = SAVE / 'official_direct_stage1'
official_dir.mkdir(parents=True, exist_ok=True)

dev_path = SAVE / 'development_comparison_with_online.csv'

assert dev_path.exists(), 'E7 results file not found.'

dev = pd.read_csv(dev_path)

pattern = (
    r'(frozen_online|full_online) '
    r'(classifier|prototype) '
    r'ConvNeXt(224|320)/CLIP224'
)

candidates = dev[
    dev['variant'].str.strip().str.fullmatch(pattern, na=False)
].copy()

print('Eligible configurations:', len(candidates))
assert len(candidates) == 8, 'Expected 8 online configurations.'

candidates = candidates.sort_values(
    ['composition_top1_HM', 'composition_top5_HM', 'variant'],
    ascending=[False, False, True]
)

best = candidates.iloc[0]

mode, scoring, res = re.fullmatch(
    pattern, best['variant'].strip()
).groups()

res = int(res)

lock = {
    'variant': str(best['variant']),
    'mode': mode,
    'scoring': scoring,
    'resolution': res,
    'epoch': ONLINE_EPOCHS,
    'development_top1_HM': float(best['composition_top1_HM']),
    'development_top5_HM': float(best['composition_top5_HM']),
    'training_images': 6905,
    'candidate_pairs': 9625,
    'gamma': 0,
    'selection_rule': 'Highest development Top-1 HM, then Top-5 HM',
    'development_csv_sha': sha(dev_path),
    'checkpoints': {}
}

print('Selected model:', lock['variant'])
print('Verifying checkpoint hashes...')

for a in ARMS:
    path = ONLINE / mode / a / 'last.pt'

    assert path.exists(), f'Missing checkpoint: {path}'

    lock['checkpoints'][a] = {
        'checkpoint_sha': sha(path),
        'parent_sha': sha(CKPTS[a])
    }

    if scoring == 'prototype':
        bank_path = ONLINE / mode / a / f'prototype_r{res}.pt'
        assert bank_path.exists()
        lock['checkpoints'][a]['bank_sha'] = sha(bank_path)

    print(a, 'verified')

lock_path = official_dir / 'official_protocol_lock.json'

if lock_path.exists():
    assert json.loads(lock_path.read_text()) == lock, \
        'Existing official evaluation protocol differs.'
else:
    atomic_json(lock, lock_path)

print('\nLOCKED CONFIGURATION')
print(json.dumps(lock, indent=2))

display(candidates[SHOW].round(2))

print('\nE8 completed successfully.')
print('Protocol saved to:', lock_path)


Eligible configurations: 8
Selected model: full_online classifier ConvNeXt224/CLIP224
Verifying checkpoint hashes...
crop_b1 verified
crop_clip verified
disease_clip verified
disease_small_reg verified

LOCKED CONFIGURATION
{
  "variant": "full_online classifier ConvNeXt224/CLIP224",
  "mode": "full_online",
  "scoring": "classifier",
  "resolution": 224,
  "epoch": 3,
  "development_top1_HM": 14.883720930232558,
  "development_top5_HM": 25.53373795073188,
  "training_images": 6905,
  "candidate_pairs": 9625,
  "gamma": 0,
  "selection_rule": "Highest development Top-1 HM, then Top-5 HM",
  "development_csv_sha": "8f25bb868c5aaf47ebba45148fbf3cac9386eb22ae7c94d1d3930dc4e450bb94",
  "checkpoints": {
    "crop_b1": {
      "checkpoint_sha": "86eafa479b6cfff1ea7f38d5f53e2d9bbae12e5eb21d7a5fee8792c24b3bee8f",
      "parent_sha": "64509062a54291eeee16ad89d1f0f08c5001875ae2e4bfb687df4c59d108be12"
    },
    "crop_clip": {
      "checkpoint_sha": "b97cd42cf43371932f5dbb53088db13b9f5afc5ee8a7c

,variant,epoch,seen_composition_top1,unseen_composition_top1,composition_top1_HM,composition_top5_HM,unseen_disease_top1
35,full_online classifier ConvNeXt224/CLIP224,3.0,75.97,8.25,14.88,25.53,27.00
36,full_online prototype ConvNeXt224/CLIP224,3.0,78.71,7.75,14.11,27.84,26.50
38,full_online classifier ConvNeXt320/CLIP224,3.0,76.12,7.50,13.65,25.16,27.00
39,full_online prototype ConvNeXt320/CLIP224,3.0,78.99,7.00,12.86,30.25,26.50
30,frozen_online prototype ConvNeXt224/CLIP224,3.0,77.12,6.50,11.99,28.86,25.50
33,frozen_online prototype ConvNeXt320/CLIP224,3.0,76.26,5.50,10.26,29.24,25.00
29,frozen_online classifier ConvNeXt224/CLIP224,3.0,74.96,3.75,7.14,17.91,22.25
32,frozen_online classifier ConvNeXt320/CLIP224,3.0,75.25,3.50,6.69,15.89,22.50



E8 completed successfully.
Protocol saved to: /content/drive/MyDrive/CDC_CP_ConvNeXt_CLIP_Extension_v1/official_direct_stage1/official_protocol_lock.json


In [21]:

# @title E9 · Official-test evaluation (no additional training)

official_dir = SAVE / 'official_direct_stage1'
dev_path = SAVE / 'development_comparison_with_online.csv'
lock_path = official_dir / 'official_protocol_lock.json'

assert lock_path.exists(), 'Complete E8 first.'

lock = json.loads(lock_path.read_text())
assert sha(dev_path) == lock['development_csv_sha']

mode = lock['mode']
scoring = lock['scoring']
res = lock['resolution']

official_frames = {
    'seen': st.read_labelled_csv(
        SOURCE / 'DPD_Small_test_seen.csv'
    ),
    'unseen': st.read_labelled_csv(
        SOURCE / 'DPD_Small_test_unseen.csv'
    )
}

assert len(official_frames['seen']) == 2000
assert len(official_frames['unseen']) == 1448

dev_names = set().union(
    *(set(df.filename) for df in FRAMES.values())
)

for split, df in official_frames.items():
    assert not (set(df.filename) & dev_names)
    assert set(df.crop_id) <= set(CROP_IDS)
    assert set(df.disease_id) <= set(DISEASE_IDS)
    assert all((IMAGES / str(f)).is_file() for f in df.filename)

assert not (
    set(official_frames['seen'].filename)
    & set(official_frames['unseen'].filename)
)

official_logs = {}

for a in ARMS:
    print('Evaluating:', a)

    path = ONLINE / mode / a / 'last.pt'
    expected = lock['checkpoints'][a]

    assert sha(path) == expected['checkpoint_sha']
    assert sha(CKPTS[a]) == expected['parent_sha']

    state = torch.load(
        path, map_location='cpu', weights_only=True
    )

    assert state['epoch'] == ONLINE_EPOCHS
    assert state['sha'] == PROVENANCE[a]['sha256']

    base = base_model(a)

    if mode == 'full_online':
        base.load_state_dict(state['base'])

    base.eval()

    head = make_head(a, 'deep', state['dim'])
    head.load_state_dict(state['head'])
    head.eval()

    if scoring == 'prototype':
        bank_path = ONLINE / mode / a / f'prototype_r{res}.pt'
        assert sha(bank_path) == expected['bank_sha']

        bank_state = torch.load(
            bank_path, map_location='cpu', weights_only=True
        )

        assert bank_state['sha'] == state['sha']
        assert bank_state['epoch'] == state['epoch']
        assert bank_state['resolution'] == (
            224 if is_clip(a) else res
        )

        bank = bank_state['bank']

    r = 224 if is_clip(a) else res

    for split, df in official_frames.items():
        part = cache_part(base, a, df, False, r)
        logits, embeddings = head_outputs(head, part['f'])

        if scoring == 'prototype':
            logits = (embeddings @ bank.T) / PROTOTYPE_TAU

        official_logs[a, split] = flip_logp(logits).numpy()

        print(a, split, 'completed')
        del part, logits, embeddings

    del base, head, state
    gc.collect()
    torch.cuda.empty_cache()

metrics = {}

for split, df in official_frames.items():
    c, d = fuse(official_logs, split)

    result = st.composition_metrics(
        c, d,
        df.crop_id.to_numpy(),
        df.disease_id.to_numpy()
    )

    metrics[split] = {
        k: v * 100 for k, v in result.items()
    }

rows = []

for task in ['crop', 'disease', 'composition']:
    for k in [1, 5]:
        key = f'{task}_top{k}'

        seen = metrics['seen'][key]
        unseen = metrics['unseen'][key]

        rows.append({
            'Task': task,
            'Top-K': k,
            'Seen (%)': seen,
            'Unseen (%)': unseen,
            'HM (%)': st.harmonic(seen, unseen)
        })

official_results = pd.DataFrame(rows)

official_results.to_csv(
    official_dir / 'official_results.csv',
    index=False
)

np.savez_compressed(
    official_dir / 'official_predictions.npz',
    **{
        f'{a}_{split}': values
        for (a, split), values in official_logs.items()
    }
)

display(official_results.round(2))

print('Selected model:', lock['variant'])
print('Training images:', lock['training_images'])
print('Results saved to:', official_dir)
print('E9 COMPLETED SUCCESSFULLY')


Evaluating: crop_b1
crop_b1 seen completed
crop_b1 unseen completed
Evaluating: crop_clip
crop_clip seen completed
crop_clip unseen completed
Evaluating: disease_clip
disease_clip seen completed
disease_clip unseen completed
Evaluating: disease_small_reg
disease_small_reg seen completed
disease_small_reg unseen completed


,Task,Top-K,Seen (%),Unseen (%),HM (%)
0,crop,1,89.30,71.48,79.40
1,crop,5,94.35,83.22,88.44
2,disease,1,65.50,11.05,18.91
3,disease,5,79.45,18.92,30.57
4,composition,1,62.30,6.70,12.10
5,composition,5,74.90,12.02,20.71


Selected model: full_online classifier ConvNeXt224/CLIP224
Training images: 6905
Results saved to: /content/drive/MyDrive/CDC_CP_ConvNeXt_CLIP_Extension_v1/official_direct_stage1
E9 COMPLETED SUCCESSFULLY


In [20]:

official_dir = SAVE / 'official_direct_stage1'
dev_path = SAVE / 'development_comparison_with_online.csv'

assert (official_dir / 'official_protocol_lock.json').exists(), \
    'Complete E8 successfully before running E9.'


## How to interpret the experiment

- Original ensemble 224 vs ConvNeXt320/CLIP224: resolution only, no weight changes.
- Frozen shallow CE vs frozen deep CE: projection depth (and capacity) under matched cached-feature training.
- Frozen deep CE vs frozen deep CE+SupCon: supervised contrastive objective; same architecture, fixed epoch and features.
- Deep SupCon classifier vs prototype: inference rule, same trained concept heads and test features. Prototype banks built exclusively on 6,905 fitting images.
- Original + prototypes: predeclared 50/50 score-fusion extension; not evidence of a learned pair interaction.
- Matched online frozen vs full: fine-tuning comparison, same parent and fixed continuation budget. Compare within identical resolution and scoring rule.

The reference is the Stage 1 development ensemble, not the published Stage 2 official 16.47% HM. Do not present development improvements as official-test improvements. Historical official test has already been inspected; a newly selected variant requires a locked protocol, independent evaluation or clearly labelled post-hoc testing. Separate concept heads and SupCon are inductive biases, not proof of disentanglement. Report seen, unseen, HM, disease accuracy, and the results that fail as well as those that improve.